# Agent control loop

These are the judgments an agent makes around a turn: which model is enough, which tool fits, whether the call is safe, and whether the task is actually finished.

Each section is one decision. Python owns the branch. Jev only answers the question.


In [1]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


Jev model: jev-latest
Jev key set: True
OpenAI key set: True


## 1. Model routing

A product question can use a cheap model. A missing paid order needs a stronger one. This cell prints the choice. It does not call OpenAI.


In [2]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    samples = {
        "simple": ticket("T-270")["body"],
        "hard": ticket("T-260")["body"],
    }
    questions = {
        "route": Choice(
            instructions="Which model should handle this message?",
            criteria={
                "fast": "A lookup, a product fact, or a short status answer.",
                "powerful": "Several steps, an angry shopper, or a judgment about money.",
            },
        )
    }
    for name, text in samples.items():
        response = ask(text, questions)
        show(response)
        print(name, "->", response.choices["route"].choice)


model: jev-1.13.0
  choice route: fast  (confidence 1.00)
simple -> fast


model: jev-1.13.0
  choice route: powerful  (confidence 0.98)
hard -> powerful


**What you should see.** `simple` should land on fast. `hard` (the missing tent, written in all caps) should land on powerful.


## 2. Tool-risk gating

Same question the Auto Mode middleware asks: does this call only read, or does it change something? A risky call is reviewed. A read can pass.


In [3]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    calls = load_json("tools.json")["calls"]
    questions = {
        "risky": Noul(
            instructions="Does this call change, refund, or delete data, rather than only read it?",
            criteria=NoulCriteria(
                true="The call writes, deletes, refunds, cancels, or sends a message.",
                false="The call only reads data the user can look at.",
            ),
        )
    }
    for call in calls:
        response = ask({"request": call["request"], "call": call}, questions)
        show(response)
        route = "review" if response.nouls["risky"].noul >= 0.5 else "pass"
        print(call["name"], "->", route)


model: jev-1.13.0
  noul   risky: 0.03
lookup_order -> pass


model: jev-1.13.0
  noul   risky: 0.93
refund_order -> review


**What you should see.** `lookup_order` should pass. `refund_order` on a 'where is my order' request should go to review.


## 3. Tool selection

Pick a tool from a short roster, and also ask whether a tool is needed at all. The roster here is six tools. The same shape works for a longer list.


In [4]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    roster = load_json("tools.json")["roster"]
    request = ticket("T-270")["body"]
    wide = ask(
        request,
        {
            "which": Choice(instructions="Which tool fits this request?", criteria=roster),
            "needs_tool": Noul(instructions="Does answering this require a tool, rather than a general sentence?"),
        },
    )
    show(wide)
    if wide.nouls["needs_tool"].noul < 0.35:
        print("route: answer directly")
    else:
        ranked = sorted(wide.choices["which"].probabilities.items(), key=lambda item: -item[1])[:3]
        shortlist = [name for name, _score in ranked]
        fine = ask(
            request,
            {
                "which": Choice(
                    instructions="Which of these tools is the right one?",
                    criteria={name: roster[name] for name in shortlist},
                )
            },
        )
        show(fine)
        print("route:", fine.choices["which"].choice)


model: jev-1.13.0
  noul   needs_tool: 0.72
  choice which: search_products  (confidence 1.00)


model: jev-1.13.0
  choice which: search_products  (confidence 1.00)
route: search_products


**What you should see.** The product question about the 40L pack should point at `search_products`.


## 4. Check a tool call before it runs

Split 'was this call right?' into two yes/no questions. A wrong tool and a wrong argument then show up as different failures.


In [5]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    schema = load_json("tools.json")["schema"]
    questions = {
        "tool_is_relevant": Noul(instructions="Is `call.name` a fitting tool for `request`?"),
        "args_match_request": Noul(instructions="Do the values in `call.arguments` match what `request` asked for?"),
    }
    for call in load_json("tools.json")["calls"]:
        response = ask(
            {"request": call["request"], "schema": schema, "call": call},
            questions,
        )
        show(response)
        failing = [name for name, answer in response.nouls.items() if answer.noul < 0.5]
        print(call["name"], "->", "reject" if failing else "run", failing)


model: jev-1.13.0
  noul   tool_is_relevant: 0.96
  noul   args_match_request: 0.97
lookup_order -> run []


model: jev-1.13.0
  noul   tool_is_relevant: 0.02
  noul   args_match_request: 0.79
refund_order -> reject ['tool_is_relevant']


**What you should see.** The lookup of A-118 should run. The refund call, made for a tracking question, should be rejected.


## 5. Fill closed-set tool arguments

Most arguments are already a fixed list. Jev picks the label. Free text stays in code. Confidence of the call is the weakest judgment, not an average.


In [6]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    text = load_json("tools.json")["issue_requests"][0]
    print(text)
    response = ask(
        text,
        {
            "__tool__": Choice(
                instructions="What is the user asking the assistant to do?",
                criteria={"create_issue": "File a new issue", "list_issues": "Show existing issues"},
            ),
            "repo": Choice(
                instructions="Which repository is this about?",
                criteria={"checkout": "Payments and checkout", "catalog": "Product pages and search"},
            ),
            "priority": Score(
                instructions="How urgent does the report sound?",
                criteria=["Low, no rush", "Normal", "Blocking or urgent"],
            ),
            "label_bug": Noul(instructions="Does the request describe a bug?"),
        },
    )
    show(response)
    levels = ["low", "normal", "high"]
    priority = levels[min(2, max(0, round(response.scores["priority"].score)))]
    weakest = min(response.choices["__tool__"].confidence, response.choices["repo"].confidence)
    print({"tool": response.choices["__tool__"].choice, "repo": response.choices["repo"].choice, "priority": priority, "confidence": round(weakest, 2)})


File a bug on the checkout repo. Customers get 500s and it is blocking.


model: jev-1.13.0
  noul   label_bug: 0.96
  choice __tool__: create_issue  (confidence 1.00)
  choice repo: checkout  (confidence 1.00)
  score  priority: 2.00  ~ Blocking or urgent
{'tool': 'create_issue', 'repo': 'checkout', 'priority': 'high', 'confidence': 1.0}


**What you should see.** The checkout 500s sentence should become a bug issue on the checkout repo, with high priority.


## 6. Is the task done?

Agents stop too early. Ask whether the answer actually finishes the request, and whether it still admits work is left.


In [7]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "goal_satisfied": Noul(instructions="Does `final_answer` fully satisfy `original_request`?"),
        "left_todo": Noul(instructions="Does `final_answer` say something is still pending or not done?"),
        "quality": Score(
            instructions="How complete is `final_answer` for `original_request`?",
            criteria=["Incomplete", "Partial", "Complete"],
        ),
    }
    for item in load_json("agent.json")["completions"]:
        response = ask(item, questions)
        show(response)
        done = response.nouls["goal_satisfied"].noul > 0.7 and response.nouls["left_todo"].noul < 0.4
        print("route:", "end" if done else "continue")


model: jev-1.13.0
  noul   goal_satisfied: 0.86
  noul   left_todo: 0.02
  score  quality: 1.91  ~ Complete
route: end


model: jev-1.13.0
  noul   goal_satisfied: 0.12
  noul   left_todo: 0.98
  score  quality: 0.47  ~ Incomplete
route: continue


**What you should see.** The answer that says the refund was submitted should end. The answer that says the refund is still needed should continue.


## 7. Ask a human only when it matters

A low-stakes read can proceed at modest confidence. Anything destructive needs a high bar, or a person.


In [8]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    samples = [ticket("T-118")["body"], "Delete my account and every saved address."]
    questions = {
        "action": Choice(
            instructions="What action is the user requesting?",
            criteria={
                "read_order": "See status, tracking, or a summary",
                "refund": "Get money back",
                "delete_account": "Remove the account or its data",
            },
        )
    }
    for text in samples:
        response = ask(text, questions)
        show(response)
        action = response.choices["action"]
        if action.confidence < 0.6:
            route = "clarify"
        elif action.choice == "read_order":
            route = "read"
        elif action.confidence < 0.85:
            route = "ask_human"
        else:
            route = action.choice
        print("route:", route)


model: jev-1.13.0
  choice action: read_order  (confidence 1.00)
route: read


model: jev-1.13.0
  choice action: delete_account  (confidence 1.00)
route: delete_account


**What you should see.** The tracking note should read. The delete sentence should be `delete_account` only if confidence is high; otherwise it should ask a human.


## 8. Which specialist gets this task?

A supervisor can dispatch with a Choice and a complexity Score. No chat model is required for the handoff itself.


In [9]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    response = ask(
        {"task": ticket("T-201")["body"]},
        {
            "specialist": Choice(
                instructions="Which specialist should own this task?",
                criteria={
                    "support": "Status, tracking, and simple product facts",
                    "billing": "Refunds, duplicate charges, and invoices",
                    "reviewer": "Check an answer someone already drafted",
                },
            ),
            "needs_clarification": Noul(instructions="Is the task too vague to act on without another question?"),
            "complexity": Score(
                instructions="How hard is this to resolve?",
                criteria=["Simple", "Some judgment", "Needs a person"],
            ),
        },
    )
    show(response)
    if response.nouls["needs_clarification"].noul > 0.7:
        route = "ask_requester"
    elif response.choices["specialist"].confidence < 0.5:
        route = "human_lead"
    else:
        route = response.choices["specialist"].choice
    print("route:", route)


model: jev-1.13.0
  noul   needs_clarification: 0.46
  choice specialist: billing  (confidence 0.99)
  score  complexity: 0.30  ~ Simple
route: billing


**What you should see.** The broken-poles refund should go to billing.


## 9. What to keep when you trim memory

Score each note before you compact a conversation. Drop secrets. Keep decisions.


In [10]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    agent = load_json("agent.json")
    questions = {
        "contains_decision": Noul(instructions="Does `chunk` record a decision or an agreed amount?"),
        "contains_secret": Noul(instructions="Does `chunk` contain a key, token, or password?"),
        "importance": Score(
            instructions="How important is `chunk` for `current_goal`?",
            criteria=["Noise", "Background", "Needed to finish"],
        ),
    }
    requests = [
        {"state": {"chunk": chunk, "current_goal": agent["current_goal"]}, "questions": questions}
        for chunk in agent["memory_chunks"]
    ]
    results = ask_many(requests)
    kept = []
    for chunk, response in zip(agent["memory_chunks"], results):
        show(response)
        if response.nouls["contains_secret"].noul > 0.5:
            print("drop secret")
            continue
        if response.nouls["contains_decision"].noul > 0.6 or response.scores["importance"].score > 1.4:
            kept.append(chunk)
    print("kept:", kept)


model: jev-1.13.0
  noul   contains_decision: 0.94
  noul   contains_secret: 0.03
  score  importance: 1.91  ~ Needed to finish
model: jev-1.13.0
  noul   contains_decision: 0.03
  noul   contains_secret: 0.01
  score  importance: 0.03  ~ Noise
model: jev-1.13.0
  noul   contains_decision: 0.03
  noul   contains_secret: 0.97
  score  importance: 0.13  ~ Noise
drop secret
model: jev-1.13.0
  noul   contains_decision: 0.04
  noul   contains_secret: 0.02
  score  importance: 0.01  ~ Noise
kept: ['We agreed the extra $49 charge on A-104 will be refunded.']


**What you should see.** The agreed $49 refund should be kept. The API key note should be dropped. The weather comment should not be treated as a decision.


## Optional middleware

The same two decisions ship as `ModelRouterMiddleware` and `AutoModeMiddleware`. This cell only checks that the extra builds. It does not start an agent.


In [11]:
try:
    from langchain_typesafe.experimental.middleware import AutoModeMiddleware, ModelChoice, ModelRouterMiddleware
    print("imported middleware")
except Exception as exc:
    print("middleware extra is not installed:", type(exc).__name__)
else:
    if not typesafe_ready() or not openai_ready():
        print("skipped: building the router needs both API keys, because it loads a chat model")
    else:
        try:
            router = ModelRouterMiddleware(
                choices={
                    "fast": ModelChoice(model="openai:gpt-4o-mini", criteria="Lookups and short replies."),
                    "powerful": ModelChoice(model="openai:gpt-4o-mini", criteria="Multi-step or angry cases."),
                },
                instructions="Choose the least costly model that can do the task.",
            )
            gate = AutoModeMiddleware(tools=["refund_order"])
            print(type(router).__name__, type(gate).__name__)
        except Exception as exc:
            print("middleware did not build:", type(exc).__name__, exc)


imported middleware


ModelRouterMiddleware AutoModeMiddleware


**What you should see.** You should see the middleware class names if both keys are set. With either key missing, the cell skips after a successful import.
